In [2]:
def get_monthly_returns(ticker: str, start_date: str = "2022-01-01") -> 'pd.DataFrame':
    """
    Returns monthly open, close and percentage return for a stock ticker.

    Parameters:
        ticker (str): Yahoo Finance ticker (example: TCS.NS)
        start_date (str): Start date in YYYY-MM-DD

    Returns:
        pandas.DataFrame with:
            YearMonth
            month_open
            month_close
            monthly_return_pct
    """

    import yfinance as yf
    import pandas as pd

    # Download data
    df = yf.download(ticker, start=start_date, auto_adjust=False)

    # Check if empty
    if df.empty:
        raise ValueError(f"No data returned for ticker {ticker}")

    # Fix MultiIndex columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # Reset index to access Date column
    df = df.reset_index()

    # Create YearMonth column
    df["YearMonth"] = df["Date"].dt.to_period("M")

    # Group by month
    monthly = df.groupby("YearMonth").agg(
        month_open=("Open", "first"),
        month_close=("Close", "last")
    ).reset_index()

    # Calculate return %
    monthly["monthly_return_pct"] = (
        (monthly["month_close"] - monthly["month_open"])
        / monthly["month_open"]
    ) * 100
    monthly["monthly_return_pct"] = monthly["monthly_return_pct"].round(2)

    return monthly

In [3]:
def get_multiple_stocks_monthly_returns(tickers: list, start_date="2022-01-01"):

    import pandas as pd

    all_data = []

    for ticker in tickers:

        df = get_monthly_returns(ticker, start_date)

        df["ticker"] = ticker

        all_data.append(df)

    return pd.concat(all_data, ignore_index=True)

In [4]:
tickers = [
    "ABCAPITAL.NS",
    "BHEL.NS",
    "CDSL.NS",
    "GOLDBEES.NS",
    "HCLTECH.NS",
    "JSL.NS",
    "MON100.NS",
    "NAM-INDIA.NS",
    "PNB.NS",
    "POLYCAB.NS",
    "SONACOMS.NS",
    "SOUTHBANK.NS",
    "SUMICHEM.NS",
    "UNIONBANK.NS",
    "UTIAMC.NS",
    "VBL.NS",
    "VINATIORGA.NS"
]



result = get_multiple_stocks_monthly_returns(tickers)
result.to_csv("all_stocks_monthly_returns.csv", index=False)

print(result)

Failed to get ticker 'ABCAPITAL.NS' reason: HTTPSConnectionPool(host='query2.finance.yahoo.com', port=443): Read timed out. (read timeout=10)
[*********************100%%**********************]  1 of 1 completed

1 Failed download:
['ABCAPITAL.NS']: YFTzMissingError('$%ticker%: possibly delisted; No timezone found')


ValueError: No data returned for ticker ABCAPITAL.NS

In [ ]:
def get_mf_monthly_returns(scheme_code: str, start_date="2022-01-01"):
    """
    Fetch mutual fund NAV history and calculate monthly returns

    Returns:
        YearMonth
        month_open
        month_close
        monthly_return_pct
    """

    import requests
    import pandas as pd

    # Fetch NAV history
    url = f"https://api.mfapi.in/mf/{scheme_code}"
    data = requests.get(url).json()["data"]

    df = pd.DataFrame(data)

    # Convert date format
    df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y")

    # Filter start date
    df = df[df["date"] >= start_date]

    # Convert nav to float
    df["nav"] = df["nav"].astype(float)

    # Create YearMonth
    df["YearMonth"] = df["date"].dt.to_period("M")

    # Sort properly
    df = df.sort_values("date")

    # Monthly open and close NAV
    monthly = df.groupby("YearMonth").agg(
        month_open=("nav", "first"),
        month_close=("nav", "last")
    ).reset_index()

    # Calculate return %
    monthly["monthly_return_pct"] = (
        ((monthly["month_close"] - monthly["month_open"])
         / monthly["month_open"]) * 100
    ).round(2)

    return monthly

In [ ]:
def get_multiple_mf_monthly_returns(scheme_codes, start_date="2022-01-01"):

    import pandas as pd

    all_data = []

    for code in scheme_codes:

        df = get_mf_monthly_returns(code, start_date)

        df["scheme_code"] = code

        all_data.append(df)

    return pd.concat(all_data, ignore_index=True)

In [ ]:
def save_mf_monthly_returns_to_csv(scheme_codes, start_date, file_path):

    df = get_multiple_mf_monthly_returns(scheme_codes, start_date)

    df.to_csv(file_path, index=False)

    print(f"Saved to {file_path}")

In [ ]:
scheme_codes = [
    "119533",
    "120692",
    "118796",
    "118275",
    "120166",
    "122639",
    "120843",
    "147704",
    "127042",
    "149368",
    "149185",
    "102432",
    "135800",
    "147946",
    "120251",
    "120821",
    "146513",
    "140088",
    "119648",
    "148807",
    "149838",
    "148815",
    "120620",
    "149389",
    "120684",
    "149283",
    "147622",
    "152881",
    "148726",
    "148519",
    "143341",
    "149283"
]

save_mf_monthly_returns_to_csv(
    scheme_codes,
    "2022-01-01",
    "mf_monthly_returns.csv"
)